In [ ]:
import math
import torch
import random
import numpy as np
from torch import nn, optim

from datetime import datetime
from dataclasses import dataclass, asdict
from tokenizer import TinyStoriesTokenizer
from self_attention import MultiHeadSelfAttention

# Set all seeds to make results repeatable
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
class PositionwiseFFN(nn.Module):
    """
    The position-wise FFN that follows after the self-attention
    computation.
    """

    def __init__(self, hidden_size, dropout_prob) :
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size, bias=True)
        self.fc2 = nn.Linear(hidden_size, hidden_size, bias=True)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

class Block(nn.Module):
    """
    Transformer encoder block.

    This version differs from the original version in  [Vaswani et al. NeurIPS 2017],
    and applies the LayerNorm before the self-attention, and before the FFN, as this
    has proved to be beneficial (see [Nguyen and Salazar 2019]).
    """

    def __init__(self, hidden_size, number_of_attention_heads, block_size, dropout_prob):
        super().__init__()
        att_dim = hidden_size // number_of_attention_heads
        self.attn = MultiHeadSelfAttention(hidden_size, att_dim, block_size)
        self.ffn = PositionwiseFFN(hidden_size, dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)

    def forward(self, x):
        x1 = self.ln1(x)
        x2 = x + self.dropout(self.attn(x1))
        x3 = self.ln2(x2)
        x4 = x2 + self.dropout(self.ffn(x3))
        return x4

In [ ]:
# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 4
    hidden_size: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesLM(nn.Module):

    def __init__(self, config):
        super(TinyStoriesLM, self).__init__()
        self.config = config
        self.embed =  nn.Embedding(config.vocab_size, config.hidden_size)
        self.positional = nn.Parameter(torch.randn(1, config.block_size, config.hidden_size))
        modules = [Block(config.hidden_size,\
                         config.number_of_attention_heads,\
                         config.block_size,\
                         config.dropout_prob) for _ in range(config.number_of_transformer_blocks)]
        self.transformers = nn.ModuleList(modules)
        self.final = nn.Linear(config.hidden_size, config.vocab_size)

    def forward(self, x):

        # YOUR CODE HERE
        
        return logits

    def generate(self, ids, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            ids = ids[-self.config.block_size:]  # Make sure that we don't input more that `block_size` tokens
            logits = self.forward(ids.unsqueeze(0)).squeeze(0)
            logits = logits[-1, :]  # We only want the last time step
            logits = logits / temperature
            if top_k is not None:
                # Only consider the top k classes 
                top_idx = torch.topk(logits, top_k)[1]
                mask = torch.ones(self.config.vocab_size, dtype=torch.bool)
                mask[top_idx] = False
                logits[mask] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            # Sample from the distribution and append the new token to the input
            next_id = torch.multinomial(probs, num_samples=1)
            ids = torch.cat((ids, next_id), dim=0)
        return ids
        
    @classmethod
    def load(cls, checkpoint_path, device='cpu'):
        """
        Loads a model from a checkpoint file.
        Automatically reconstructs the config and model architecture.
        """
        checkpoint = torch.load(checkpoint_path, map_location=device)
        config = Config(**checkpoint['config'])
        model = cls(config)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
    
        print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']}, iteration {checkpoint['iteration']}")
        return model


In [ ]:
from torch.utils.data import Sampler, SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 


# This class facilitates re-starting training without repeating any past training examples
class ResumableRandomSampler(Sampler):
    def __init__(self, data_source, start_index=0, seed=42):
        self.data_source = data_source
        self.start_index = start_index
        self.seed = seed
        self.generator = torch.Generator()
        self.generator.manual_seed(self.seed)
        
        # 1. Generate the full shuffled list of indices once
        self.indices = torch.randperm(len(self.data_source), generator=self.generator).tolist()

    def __iter__(self):
        # 2. Slice the indices to only include those from start_index onwards
        return iter(self.indices[self.start_index:])

    def __len__(self):
        return len(self.indices) - self.start_index

In [ ]:
# Let's have a look at some data
config = Config
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', config.block_size)
tokenizer = TinyStoriesTokenizer.load('tokenizer.json')
batch = 0
for x,y in training_dataset:
    input_ids = x.tolist()[:3]
    print( input_ids[:3] )
    decoded_input =  [tokenizer.vocab[i] for i in x.tolist()[:3]]
    decoded_target = [tokenizer.vocab[i] for i in y.tolist()[:3]]
    print( "Input:" )
    print( decoded_input )
    print( "Target:" )
    print( decoded_target )
    batch += 1
    if batch==1:
        break
    

In [ ]:
import os

# ======================= Training ======================= #

# Remove the checkpoint file if you want to train from scratch
checkpoint_path = 'last_checkpoint.pt'
RESUME_TRAINING = os.path.exists(checkpoint_path) 

seed = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print( "Running on", device )

tokenizer = TinyStoriesTokenizer.load('tokenizer.json')

if RESUME_TRAINING:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']+1}, iteration {checkpoint['iteration']})")
    config = Config(**checkpoint['config'])
    lm = TinyStoriesLM(config)
    # Sanity check
    if tokenizer.vocab_size != lm.config.vocab_size:
        print("WARNING: The tokenizer's and the model's vocab_size are different!")
    lm.load_state_dict(checkpoint['model_state_dict'])
    lm.to(device) 
else :
    config = Config()
    config.vocab_size = tokenizer.vocab_size
    lm = TinyStoriesLM(config).to(device)

lm_optimizer = torch.optim.AdamW(lm.parameters(), lr=lm.config.learning_rate, weight_decay=config.weight_decay)

if RESUME_TRAINING:
    lm_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', config.block_size)
print( "There are", len(training_dataset), "datapoints and", tokenizer.vocab_size, "token classes in the dataset" ) 

criterion = nn.CrossEntropyLoss()

lm.train()
print( datetime.now().strftime("%X"), "Training starts" )
# Can we pick up where we left off?
start_epoch = checkpoint['epoch'] if RESUME_TRAINING else 0
start_iter = checkpoint['iteration'] if RESUME_TRAINING else 0
for epoch in range(start_epoch, config.no_of_epochs) :
    iteration = start_iter if epoch == start_epoch else 0
    # Random sampling of datapoints. Note that we use a new seed for each epoch 
    # -- this will guarantee that we can resume the training anywhere without 
    # repeating any datapoints within an epoch.
    sampler = ResumableRandomSampler(training_dataset, 
                                     start_index=iteration * config.batch_size, 
                                     seed=seed+epoch) 
    training_loader = DataLoader(training_dataset, 
                                 batch_size=config.batch_size, 
                                 sampler=sampler, 
                                 num_workers=4)
    for x,y in training_loader:
        x,y = x.to(device), y.to(device)
        lm_optimizer.zero_grad()
        logits = lm(x).to(device)
        loss = criterion(logits.reshape(-1, tokenizer.vocab_size), y.reshape(-1))
        loss.backward()
        lm_optimizer.step()
        iteration += 1
        if iteration%100 == 0:
            print( datetime.now().strftime("%X"), "Epoch", epoch+1, ", iteration", iteration, " loss=", loss.detach().item())
        if iteration%10000 == 0:
            checkpoint = {
                'model_state_dict': lm.state_dict(),
                'optimizer_state_dict': lm_optimizer.state_dict(),
                'epoch': epoch,
                'iteration': iteration,
                'config': config.__dict__, # Useful so you know the architecture later
            }

            # Overwrites the same file every time to save space
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Show progress by generating a story
            prompt = "Once upon a time"
            _, start_ids = tokenizer.tokenize(prompt)
            x = torch.tensor(start_ids, dtype=torch.long, device=device)
            with torch.no_grad():
                y = lm.generate(x, max_new_tokens=100, temperature=0.8, top_k=40)
            decoded_output = [tokenizer.vocab[i] for i in y.tolist()]
            print("-" * 30)
            print("".join(decoded_output))
            print("-" * 30)

13:53:19 Epoch 1 , iteration 500  loss= 4.11009407043457
13:53:32 Epoch 1 , iteration 600  loss= 3.9573183059692383


In [ ]:
def tell_story(model, tokenizer, prompt="Once upon a time", max_tokens=100):
    device = next(model.parameters()).device
    
    # Encode prompt
    _, start_ids = tokenizer.tokenize(prompt)
    print(start_ids)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)
    
    # Generate
    with torch.no_grad():
        y = model.generate(x, max_new_tokens=max_tokens, temperature=0.8, top_k=40)
    
    # Decode and print
    decoded_output = [tokenizer.vocab[i] for i in y.tolist()]
    print("-" * 30)
    print("".join(decoded_output))
    print("-" * 30)

checkpoint_path = "last_checkpoint.pt"
tokenizer_path = "tokenizer.json"
model = TinyStoriesLM.load(checkpoint_path, device='cuda')
tokenizer = TinyStoriesTokenizer.load(tokenizer_path)
tell_story(model, tokenizer)
